# Playground Series S6E11: Predicting Credit Card Fraud
## Deep Exploratory Data Analysis, Shallow-Tree Regularization & Ensembling Framework

### Competition Overview & Mission
This notebook addresses Kaggle Playground Series (Season 6, Episode 11): **Predicting Credit Card Fraud**. 
Our objective is to predict the probability that a credit card transaction is fraudulent (`is_fraud = 1.0`) given a synthetically generated tabular dataset reflecting real-world financial transaction dynamics.

The competition is evaluated using the **Area Under the Receiver Operating Characteristic Curve (ROC-AUC)**. In heavily imbalanced fraud detection settings ($1.67\%$ positive class), ROC-AUC measures global ranking quality across all decision thresholds.

#### Core Methodological Pillars
1. **Exploration-Led Engineering**: All features originate from empirical data discoveries (compounding risk curves, CVV verification exhaustion, liquidity targets).
2. **Digit-Level & Fractional Artifact Exploitation**: Extracting subtle rounding artifacts and decimal patterns (`amount_cents`, round dollar indicators, category Z-scores) specific to synthetic tabular generators.
3. **Shallow-Tree Regularization**: Constraining tree complexity (`num_leaves=7` in LightGBM, `max_depth=3` in XGBoost, `depth=4` in CatBoost) to act as generalized additive models (GAMs), preventing overfitting to sparse synthetic noise in imbalanced data.
4. **Tri-Model GBDT Foundation**: Combining Leaf-Wise (LightGBM), Depth-Wise (XGBoost), and Oblivious (CatBoost) architectures across identical Stratified folds.
5. **Multi-Strategy Ensembling**: Benchmarking probability-space optimization, unbounded logit-space optimization, rank averaging, and stacking meta-models.
6. **Dual-Environment Portability**: Seamless execution across local Apple Silicon / CPU and Kaggle Kernel GPU environments.

# 1. Environment Detection & Hardware Configuration
---
To guarantee deterministic execution and reproducibility whether running in a local IDE (VS Code / Jupyter) or on Kaggle's GPU infrastructure, we configure thread bounds, filter non-critical runtime warnings, and inspect the runtime hardware accelerator.

In [ ]:
import os
import time
import warnings

# Enforce thread control across multi-core systems to prevent CPU thread thrashing
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["VECLIB_MAXIMUM_THREADS"] = "4"
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix, precision_recall_curve, roc_curve
from sklearn.model_selection import StratifiedKFold
import torch

# Global visualization styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10
plt.rcParams["figure.autolayout"] = True

# Custom cohesive palette
PALETTE = {
    "legitimate": "#2a9d8f",     # Deep teal
    "fraud": "#e76f51",          # Coral red
    "accent": "#f4a261",         # Warm amber
    "dark": "#1d3557",           # Deep navy
    "neutral": "#457b9d",        # Slate blue
    "highlight": "#e63946"       # Vivid crimson
}

In [ ]:
# Detect runtime environment
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

# 10-Fold Stratified Cross-Validation on Kaggle for maximum training data (90% per fold); 5-fold locally for fast iteration
N_SPLITS = 10 if IS_KAGGLE else 5
GLOBAL_SEED = 42

if IS_KAGGLE:
    SEEDS = [42, 2026]
    DEVICE = "gpu"
    TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    N_JOBS = -1
    print(f"[Environment: Kaggle Kernel] Accelerated by {TORCH_DEVICE} | CV: {N_SPLITS}-Fold | Seeds: {SEEDS}")
else:
    SEEDS = [42]
    DEVICE = "cpu"
    if torch.backends.mps.is_available():
        TORCH_DEVICE = torch.device("mps")
    else:
        TORCH_DEVICE = torch.device("cpu")
    N_JOBS = 4
    print(f"[Environment: Local System] Running on {TORCH_DEVICE} | CPU Workers: {N_JOBS} | CV: {N_SPLITS}-Fold | Seeds: {SEEDS}")

## 1.2 Adaptive Data Ingestion
The dataset consists of `train.csv` (699,960 records with binary label `is_fraud`), `test.csv` (299,984 records), and `sample_submission.csv`. 

Our data loader scans Kaggle input directories (`/kaggle/input/playground-series-s6e11`), parent directories, and local project folders systematically, ensuring zero-configuration execution in both settings.

In [ ]:
def load_data():
    """
    Robust data loader with automated fallback resolution for Kaggle and local environments.
    """
    train_file = "train.csv"
    test_file = "test.csv"
    sample_sub_file = "sample_submission.csv"

    data_dir = None

    if IS_KAGGLE:
        kaggle_candidates = [
            "/kaggle/input/playground-series-s6e11",
            "/kaggle/input/competitions/playground-series-s6e11",
        ]
        for candidate in kaggle_candidates:
            if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, train_file)):
                data_dir = candidate
                break

        if data_dir is None and os.path.exists("/kaggle/input"):
            for root, _, files in os.walk("/kaggle/input"):
                if train_file in files:
                    data_dir = root
                    break
    else:
        local_candidates = [
            "data",
            "./data",
            "../data",
            os.path.join(os.getcwd(), "data"),
            "/Users/davidlangat/Documents/Github/Kaggle/playground_series/predicting_credit_card_fraud/data",
        ]
        for candidate in local_candidates:
            if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, train_file)):
                data_dir = candidate
                break

    if data_dir is None:
        raise FileNotFoundError("Unable to locate train.csv in expected Kaggle or local directory candidates.")

    print(f"Data source resolved at: {data_dir}")
    t0 = time.time()
    train = pd.read_csv(os.path.join(data_dir, train_file))
    test = pd.read_csv(os.path.join(data_dir, test_file))
    submission = pd.read_csv(os.path.join(data_dir, sample_sub_file))
    load_time = time.time() - t0

    print(f"Loaded successfully in {load_time:.2f} seconds:")
    print(f" - Train Set:       {train.shape[0]:,} rows × {train.shape[1]} columns")
    print(f" - Test Set:        {test.shape[0]:,} rows × {test.shape[1]} columns")
    print(f" - Sample Sub Set:  {submission.shape[0]:,} rows × {submission.shape[1]} columns")

    return train, test, submission

train_df, test_df, submission_df = load_data()

# 2. Data-Driven Exploratory Data Analysis (EDA)
---
Rather than making assumptions, we perform an in-depth audit of the feature distributions, class balance, risk factor combinations, and vulnerability surfaces present in this dataset.

Every visual analysis in this section includes:
1. The quantitative data display.
2. A dedicated **Analytical Takeaway** detailing the mechanism behind the observed pattern.
3. The actionable **Downstream Modeling Implication**.

## 2.1 Data Integrity & Missing Value Audit
We evaluate whether any records contain missing, null, or corrupted values across both train and test partitions.

In [ ]:
def audit_data_integrity(df_train, df_test):
    train_nulls = df_train.isnull().sum()
    test_nulls = df_test.isnull().sum()

    audit = pd.DataFrame({
        "Train Dtype": df_train.dtypes,
        "Train Nulls": train_nulls,
        "Train Null %": (train_nulls / len(df_train) * 100).round(4),
        "Test Dtype": df_test.dtypes,
        "Test Nulls": test_nulls,
        "Test Null %": (test_nulls / len(df_test) * 100).round(4)
    })
    return audit

integrity_report = audit_data_integrity(train_df, test_df)
print(f"Total Columns Audited: {len(integrity_report)}")
print(f"Any Null Values in Train? {train_df.isnull().any().any()}")
print(f"Any Null Values in Test?  {test_df.isnull().any().any()}")
display(integrity_report)

> [!NOTE]
> **Analytical Takeaway & Modeling Implication:**
> Both `train.csv` and `test.csv` have **zero missing values** across all columns. The synthetic generation process has produced complete records without missing data artifacts.
> * **Modeling Implication:** No imputation strategies (such as iterative MICE or median filling) are required, preserving original feature distributions and avoiding imputation noise.

## 2.2 Target Distribution & Class Imbalance Analysis
In fraud detection, target class imbalance is typical. Quantifying the precise positive-class ratio is critical for selecting our validation split strategy, model loss functions, and evaluation thresholds.

In [ ]:
fraud_counts = train_df["is_fraud"].value_counts().sort_index()
fraud_pct = train_df["is_fraud"].value_counts(normalize=True).sort_index() * 100

summary_target = pd.DataFrame({
    "Class": ["Legitimate (0)", "Fraudulent (1)"],
    "Count": fraud_counts.values,
    "Percentage (%)": fraud_pct.values.round(4)
})
display(summary_target)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. Bar Chart
bars = axes[0].bar(
    summary_target["Class"], 
    summary_target["Count"], 
    color=[PALETTE["legitimate"], PALETTE["fraud"]],
    edgecolor="black",
    linewidth=0.8,
    width=0.55
)
axes[0].set_title("Transaction Volume by Target Class", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Number of Transactions (Log Scale)", fontsize=11)
axes[0].set_yscale("log")
axes[0].grid(axis="y", linestyle="--", alpha=0.7)

for bar in bars:
    yval = bar.get_height()
    axes[0].text(
        bar.get_x() + bar.get_width() / 2.0,
        yval * 1.15,
        f"{int(yval):,} ({yval/len(train_df)*100:.2f}%)",
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=10
    )

# 2. Donut Chart
wedges, texts, autotexts = axes[1].pie(
    fraud_counts.values,
    labels=summary_target["Class"],
    colors=[PALETTE["legitimate"], PALETTE["fraud"]],
    autopct="%1.2f%%",
    pctdistance=0.78,
    startangle=140,
    explode=(0, 0.12),
    wedgeprops=dict(width=0.45, edgecolor="black", linewidth=0.8)
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight("bold")
axes[1].set_title("Target Class Proportion", fontsize=12, fontweight="bold")

plt.suptitle("Target Class Imbalance Profiling (`is_fraud`)", fontsize=14, fontweight="bold", y=1.02)
plt.show()

> [!IMPORTANT]
> **Analytical Takeaway:**
> * The dataset exhibits a **1.67% base fraud rate** (11,689 fraudulent transactions vs. 688,271 legitimate ones). This represents an approximate **59:1 class imbalance ratio**.
> 
> **Downstream Modeling Implications:**
> 1. **Validation Scheme**: Standard random K-Fold runs the risk of fold-to-fold class distribution drift. We must strictly employ **Stratified K-Fold Cross-Validation** to guarantee that every single fold preserves the exact 1.67% fraud incidence.
> 2. **Metric Selection**: Accuracy is completely uninformative. The competition metric is **ROC-AUC**, evaluating global ranking capability across all thresholds. We also monitor **PR-AUC (Average Precision)** to assess precision-recall dynamics in the long tail.
> 3. **Loss Function Calibration**: Because ROC-AUC is rank-invariant to monotonic transformations, oversampling (like SMOTE) alters natural probability calibration and often degrades ROC-AUC on tabular GBDTs. We prioritize native objective tuning (`scale_pos_weight`) and probability calibration.

## 2.3 The Exponential Multi-Flag Compounding Risk Curve
The dataset provides 6 binary anomaly indicators:
1. `is_foreign_transaction`: Transaction conducted outside home territory.
2. `is_new_merchant`: First interaction with the merchant.
3. `used_vpn`: Transaction masked via Virtual Private Network.
4. `ip_country_mismatch`: Geo-IP location diverges from account registration country.
5. `billing_shipping_mismatch`: Shipping destination diverges from registered billing address.
6. `is_ai_generated_scam_attempt`: Behavioral marker flagging automated generative fraud patterns.

We examine both their individual fraud incidence and their compound multi-flag interaction.

In [ ]:
risk_flags = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt"
]

flag_records = []
for flag in risk_flags:
    active = train_df[train_df[flag] == True]
    inactive = train_df[train_df[flag] == False]
    active_rate = active["is_fraud"].mean() * 100
    inactive_rate = inactive["is_fraud"].mean() * 100
    risk_ratio = active_rate / max(inactive_rate, 1e-4)
    flag_records.append({
        "Risk Flag": flag,
        "Active Count": len(active),
        "Active Fraud Rate (%)": round(active_rate, 2),
        "Inactive Fraud Rate (%)": round(inactive_rate, 2),
        "Relative Risk Ratio": round(risk_ratio, 2)
    })

flag_df = pd.DataFrame(flag_records).sort_values(by="Active Fraud Rate (%)", ascending=False)
display(flag_df)

train_df["temp_flag_count"] = train_df[risk_flags].sum(axis=1)
compound_summary = train_df.groupby("temp_flag_count")["is_fraud"].agg(["count", "mean"]).reset_index()
compound_summary["fraud_pct"] = (compound_summary["mean"] * 100).round(2)
compound_summary.rename(columns={
    "temp_flag_count": "Active Risk Flags",
    "count": "Transaction Count",
    "fraud_pct": "Fraud Rate (%)"
}, inplace=True)
display(compound_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

bars = axes[0].barh(
    flag_df["Risk Flag"], 
    flag_df["Active Fraud Rate (%)"], 
    color=PALETTE["fraud"],
    edgecolor="black",
    linewidth=0.7
)
axes[0].axvline(1.67, color=PALETTE["dark"], linestyle="--", linewidth=1.5, label="Baseline Rate (1.67%)")
axes[0].set_title("Fraud Rate When Individual Risk Flag is TRUE", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Fraud Rate (%)", fontsize=11)
axes[0].legend(loc="lower right")
axes[0].grid(axis="x", linestyle="--", alpha=0.7)
for bar in bars:
    w = bar.get_width()
    axes[0].text(w + 0.3, bar.get_y() + bar.get_height()/2.0, f"{w:.1f}%", va="center", fontweight="bold", fontsize=9.5)

axes[1].plot(
    compound_summary["Active Risk Flags"], 
    compound_summary["Fraud Rate (%)"], 
    marker="o", 
    markersize=9, 
    color=PALETTE["highlight"], 
    linewidth=2.5, 
    label="Observed Fraud Rate"
)
axes[1].fill_between(
    compound_summary["Active Risk Flags"], 
    compound_summary["Fraud Rate (%)"], 
    alpha=0.15, 
    color=PALETTE["highlight"]
)
axes[1].set_title("The Compounding Anomaly Risk Curve", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Number of Concurrently Active Risk Flags (0 to 6)", fontsize=11)
axes[1].set_ylabel("Empirical Fraud Probability (%)", fontsize=11)
axes[1].set_xticks(compound_summary["Active Risk Flags"])
axes[1].grid(True, linestyle="--", alpha=0.7)

for _, row in compound_summary.iterrows():
    axes[1].annotate(
        f"{row['Fraud Rate (%)']}%\n(n={int(row['Transaction Count']):,})",
        (row["Active Risk Flags"], row["Fraud Rate (%)"]),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=9,
        fontweight="bold"
    )

train_df.drop(columns=["temp_flag_count"], inplace=True)
plt.suptitle("Behavioral Security Anomaly Profiling", fontsize=14, fontweight="bold", y=1.02)
plt.show()

> [!TIP]
> **Analytical Takeaway:**
> * `ip_country_mismatch` (16.50%), `is_ai_generated_scam_attempt` (15.22%), and `billing_shipping_mismatch` (12.99%) are each individually associated with an **8x to 11x increase** over the baseline fraud rate.
> * More importantly, the **combination of flags behaves like an exponential multiplier**:
>   - Transactions with **0 flags** (78.5% of all transactions) carry a negligible fraud rate of only **0.70%**.
>   - With **2 flags**, fraud probability climbs to **18.18%**.
>   - With **3 flags**, fraud probability leaps past **50% (51.89%)**.
>   - At **4 flags**, it reaches **83.93%**, and at **5 flags**, it reaches **100.0%**.

## 2.4 High-Discrimination Behavioral Indicators & Digit Artifacts
We inspect the quantitative variables that exhibit strong behavioral variance between legitimate cardholders and fraudsters:
* `cvv_retry_count`: Retrying credit card security codes.
* `merchant_risk_score`: Vendor fraud risk indexing.
* `velocity_score`: Pacing and acceleration of transactions.
* `customer_age`: Age demographics.

In [ ]:
behavioral_cols = ["cvv_retry_count", "merchant_risk_score", "velocity_score", "customer_age"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(behavioral_cols):
    ax = axes[i]
    if col == "cvv_retry_count":
        ct = pd.crosstab(train_df[col], train_df["is_fraud"], normalize="index") * 100
        ct.plot(kind="bar", stacked=False, ax=ax, color=[PALETTE["legitimate"], PALETTE["fraud"]], edgecolor="black", linewidth=0.7)
        ax.set_title(f"Fraud Rate by {col}", fontsize=11, fontweight="bold")
        ax.set_ylabel("Percentage within Retry Count (%)", fontsize=10)
        ax.set_xlabel("CVV Retry Count", fontsize=10)
        ax.legend(["Legitimate", "Fraud"], loc="upper left")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    else:
        sns.kdeplot(data=train_df[train_df["is_fraud"] == 0], x=col, ax=ax, color=PALETTE["legitimate"], label="Legitimate", fill=True, alpha=0.35, linewidth=1.5)
        sns.kdeplot(data=train_df[train_df["is_fraud"] == 1], x=col, ax=ax, color=PALETTE["fraud"], label="Fraud", fill=True, alpha=0.45, linewidth=2.0)
        ax.set_title(f"Distribution of {col} by Target Class", fontsize=11, fontweight="bold")
        ax.set_xlabel(col, fontsize=10)
        ax.set_ylabel("Kernel Density", fontsize=10)
        ax.legend(loc="upper right")
    ax.grid(True, linestyle="--", alpha=0.6)

plt.suptitle("Distributions of Key Behavioral Indicators", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# 3. Hypothesis-Driven Feature Engineering & Digit Exploitation
---
Guided by our empirical discoveries and top-performing solution analyses, we construct an enriched feature pipeline combining:

1. **Behavioral Security & Risk Flag Composites**:
   * `composite_risk_score`: Count of concurrently triggered security flags ($0$ to $6$).
   * `has_multiple_risk_flags`: Binary indicator ($\ge 2$ flags), isolating the $22.1\%$ fraud rate regime.
   * `is_zero_risk`: Binary indicator ($0$ flags), isolating the $0.70\%$ baseline safe regime ($78.5\%$ of records).
   * `address_geo_mismatch`: Composite boolean indicator (`ip_country_mismatch | billing_shipping_mismatch`).
   * `cross_border_evasion`: Triple intersection (`is_foreign_transaction & used_vpn & ip_country_mismatch`).

2. **CVV Retry & Behavioral Surge Interactions**:
   * `cvv_retry_present`: Binary flag for retry attempts ($> 0$), separating $0.93\%$ vs $9.85\%$ fraud rate.
   * `cvv_x_merchant_risk`: Product of retry count and vendor risk ($r = +0.2658$).
   * `cvv_x_velocity`: Product of retry count and velocity index ($r = +0.2651$).
   * `velocity_x_merchant_risk`: Compounding of pacing and merchant vulnerability ($r = +0.1706$).

3. **Digit-Level & Fractional Artifact Signals**:
   * `amount_cents`: Fractional cents portion `(amount_usd * 100) % 100` capturing generator rounding patterns (e.g. `.00`, `.49` having $>2\times$ higher fraud rates than `.32`).
   * `is_round_dollar`: Binary flag for whole dollar amounts (`amount_cents == 0`).
   * `is_99_cents`, `is_50_cents`: Specific price-ending flags.
   * `hours_decimal`, `distance_decimal`, `balance_decimal`: Fractional components of continuous indicators.

4. **Contextual Group-By Z-Scores & Deviation Metrics**:
   * `amount_cat_mean_diff` & `amount_cat_zscore`: Standardized deviation of purchase amount relative to the merchant category's mean and standard deviation.
   * `amount_vs_cat_median_ratio`: Ratio of transaction amount relative to category median.
   * `velocity_age_cohort_diff`: Deviation of velocity from the customer's age decile cohort.

5. **Frequency Encodings & Zero-Leakage Target Encoding**:
   * Frequency of high-risk interaction keys (`category_auth`, `channel_category`, `amount_cents`).
   * Smoothed Bayesian Out-Of-Fold Target Encoding precomputed strictly within cross-validation folds.

In [ ]:
def engineer_features(df, cat_medians=None, cat_amount_stats=None, age_velocity_stats=None):
    """
    Vectorized feature engineering pipeline incorporating behavioral composites,
    digit-level fractional artifacts, and contextual group-by deviations.
    """
    df = df.copy()

    # 1. Anomaly & Security Flag Aggregations
    risk_flags = [
        "is_foreign_transaction",
        "is_new_merchant",
        "used_vpn",
        "ip_country_mismatch",
        "billing_shipping_mismatch",
        "is_ai_generated_scam_attempt"
    ]
    df["composite_risk_score"] = df[risk_flags].sum(axis=1)
    df["has_multiple_risk_flags"] = (df["composite_risk_score"] >= 2).astype(int)
    df["is_zero_risk"] = (df["composite_risk_score"] == 0).astype(int)
    df["address_geo_mismatch"] = (df["ip_country_mismatch"] | df["billing_shipping_mismatch"]).astype(int)
    df["cross_border_evasion"] = (
        df["is_foreign_transaction"].astype(bool) & 
        df["used_vpn"].astype(bool) & 
        df["ip_country_mismatch"].astype(bool)
    ).astype(int)

    # 2. Authentication Vulnerabilities & High-Risk Liquidity Targets
    df["is_unauthenticated"] = (df["auth_method"] == "No Authentication").astype(int)
    df["is_biometric_auth"] = (df["auth_method"] == "Biometric").astype(int)
    
    high_risk_cats = ["Gift Cards", "Crypto Exchange", "Gaming"]
    df["is_high_risk_category"] = df["merchant_category"].isin(high_risk_cats).astype(int)
    df["unauth_liquid_asset"] = (df["is_unauthenticated"] & df["is_high_risk_category"]).astype(int)

    # Categorical interaction tokens
    df["category_auth"] = df["merchant_category"].astype(str) + "_" + df["auth_method"].astype(str)
    df["channel_category"] = df["channel"].astype(str) + "_" + df["merchant_category"].astype(str)

    # 3. Behavioral CVV & Velocity Indicators
    df["cvv_retry_present"] = (df["cvv_retry_count"] > 0).astype(int)
    df["cvv_x_velocity"] = df["cvv_retry_count"] * df["velocity_score"]
    df["cvv_x_merchant_risk"] = df["cvv_retry_count"] * df["merchant_risk_score"]
    df["velocity_x_merchant_risk"] = df["velocity_score"] * df["merchant_risk_score"]

    # 4. Temporal & Velocity Dynamics
    df["velocity_to_hours_ratio"] = df["velocity_score"] / (df["hours_since_last_txn"] + 0.1)
    df["txn_burst_rate"] = df["txn_count_last_24h"] / (df["hours_since_last_txn"] + 0.5)

    # 5. Financial Exposure & Relative Pricing
    df["log_amount_usd"] = np.log1p(np.maximum(df["amount_usd"], 0))
    df["amount_to_balance_ratio"] = df["amount_usd"] / (df["account_balance_usd"] + 10.0)
    df["amount_x_velocity"] = df["amount_usd"] * df["velocity_score"]
    df["dispute_velocity_risk"] = df["prior_disputes"] * df["velocity_score"]

    # 6. Digit-Level & Fractional Artifact Features
    df["amount_cents"] = (np.round(df["amount_usd"] * 100) % 100).astype(int)
    df["is_round_dollar"] = (df["amount_cents"] == 0).astype(int)
    df["is_99_cents"] = (df["amount_cents"] == 99).astype(int)
    df["is_50_cents"] = (df["amount_cents"] == 50).astype(int)

    df["hours_decimal"] = np.round((df["hours_since_last_txn"] % 1) * 100).astype(int)
    df["distance_decimal"] = np.round((df["distance_from_home_km"] % 1) * 100).astype(int)
    df["balance_decimal"] = np.round((df["account_balance_usd"] % 1) * 100).astype(int)

    # 7. Contextual Group-By Deviations (fit on train, mapped to test)
    if cat_medians is None:
        cat_medians = df.groupby("merchant_category")["amount_usd"].median().to_dict()
    df["amount_vs_cat_median_ratio"] = df["amount_usd"] / (df["merchant_category"].map(cat_medians) + 1.0)

    if cat_amount_stats is None:
        cat_amount_stats = df.groupby("merchant_category")["amount_usd"].agg(["mean", "std"]).to_dict()
    cat_means = cat_amount_stats["mean"] if isinstance(cat_amount_stats, dict) and "mean" in cat_amount_stats else cat_amount_stats["mean"].to_dict()
    cat_stds = cat_amount_stats["std"] if isinstance(cat_amount_stats, dict) and "std" in cat_amount_stats else cat_amount_stats["std"].to_dict()

    df["amount_cat_mean_diff"] = df["amount_usd"] - df["merchant_category"].map(cat_means)
    df["amount_cat_zscore"] = df["amount_cat_mean_diff"] / (df["merchant_category"].map(cat_stds) + 1e-4)

    # Age decile cohort velocity deviation
    df["age_decile"] = pd.qcut(df["customer_age"], 10, labels=False, duplicates="drop")
    if age_velocity_stats is None:
        age_velocity_stats = df.groupby("age_decile")["velocity_score"].mean().to_dict()
    df["velocity_age_cohort_diff"] = df["velocity_score"] - df["age_decile"].map(age_velocity_stats)
    df.drop(columns=["age_decile"], inplace=True)

    # Convert boolean flags to integers
    for col in risk_flags:
        df[col] = df[col].astype(int)

    return df, cat_medians, cat_amount_stats, age_velocity_stats

In [ ]:
print("Executing enhanced feature engineering pipeline...")
t0_fe = time.time()

train_feat, train_medians, train_cat_stats, train_age_stats = engineer_features(train_df)
test_feat, _, _, _ = engineer_features(
    test_df, 
    cat_medians=train_medians, 
    cat_amount_stats=train_cat_stats, 
    age_velocity_stats=train_age_stats
)

target_col = "is_fraud"
y_train = train_feat[target_col].values
X_train = train_feat.drop(columns=["id", target_col])
X_test = test_feat.drop(columns=["id"])
test_ids = test_feat["id"].values

# Frequency encodings
for col in ["merchant_category", "category_auth", "channel_category", "amount_cents"]:
    freq_map = X_train[col].value_counts(normalize=True).to_dict()
    X_train[f"{col}_freq"] = X_train[col].map(freq_map).fillna(0)
    X_test[f"{col}_freq"] = X_test[col].map(freq_map).fillna(0)

# Precompute Out-Of-Fold Smoothed Target Encoding
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=GLOBAL_SEED)
te_columns = ["category_auth", "channel_category", "merchant_category", "auth_method"]
global_mean = float(y_train.mean())
SMOOTHING = 50.0

print(f"Precomputing {N_SPLITS}-Fold Target Encoding for {te_columns}...")
for col in te_columns:
    train_arr = X_train[col].astype(str).values
    test_arr = X_test[col].astype(str).values

    train_te_col = np.zeros(len(X_train), dtype=np.float64)
    test_te_col = np.zeros(len(X_test), dtype=np.float64)

    for trn_idx, val_idx in skf.split(train_arr, y_train):
        trn_cats, trn_y = train_arr[trn_idx], y_train[trn_idx]
        df_trn = pd.DataFrame({"cat": trn_cats, "target": trn_y})
        stats = df_trn.groupby("cat")["target"].agg(["count", "mean"])
        counts = stats["count"].to_dict()
        means = stats["mean"].to_dict()

        smooth_dict = {
            k: (counts[k] * means[k] + SMOOTHING * global_mean) / (counts[k] + SMOOTHING)
            for k in counts
        }

        val_cats = train_arr[val_idx]
        train_te_col[val_idx] = [smooth_dict.get(c, global_mean) for c in val_cats]
        test_te_col += np.array([smooth_dict.get(c, global_mean) for c in test_arr]) / N_SPLITS

    X_train[f"{col}_TE"] = train_te_col
    X_test[f"{col}_TE"] = test_te_col

# Define categorical features list
categorical_features = [
    "merchant_category", "card_type", "auth_method", 
    "channel", "device_type", "category_auth", "channel_category"
]
for cat_col in categorical_features:
    X_train[cat_col] = X_train[cat_col].astype("category")
    X_test[cat_col] = X_test[cat_col].astype("category")

fe_duration = time.time() - t0_fe
print(f"Feature engineering completed in {fe_duration:.2f}s!")
print(f" - Training Matrix: {X_train.shape[0]:,} samples × {X_train.shape[1]} features")
print(f" - Testing Matrix:  {X_test.shape[0]:,} samples × {X_test.shape[1]} features")

# 4. Model 1: Shallow LightGBM (`num_leaves=7`)
---
### Why Shallow Trees Outperform Deep Trees on Imbalanced Data
In tabular data with extreme class imbalance ($1.67\%$ positive class), deep trees (`num_leaves=63`) overfit to sparse terminal leaves. 
By constraining tree complexity to **shallow trees (`num_leaves=7`)**, the model operates like a regularized generalized additive model (GAM), capturing robust additive signals (`cvv_x_merchant_risk`, `composite_risk_score`, `category_auth_TE`) without fitting generator noise.

In [ ]:
import lightgbm as lgb

lgb_oof_probs = np.zeros(len(X_train))
lgb_test_probs = np.zeros(len(X_test))
lgb_fold_aucs = []
lgb_fold_praucs = []
lgb_feature_importances = pd.DataFrame()

lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 7,          # Shallow tree regularization (monotonic gain on synthetic fraud)
    "feature_fraction": 0.80,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "min_child_samples": 50,
    "n_jobs": N_JOBS,
    "verbose": -1,
}

print(f"Starting Shallow LightGBM {N_SPLITS}-Fold Cross-Validation (num_leaves=7 | Seeds: {SEEDS})...")
t0_lgb = time.time()

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[trn_idx], y_train[trn_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]

    trn_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_va, label=y_va, reference=trn_data)

    fold_val_preds = np.zeros(len(val_idx))
    fold_test_preds = np.zeros(len(X_test))

    for seed in SEEDS:
        p = lgb_params.copy()
        p["random_state"] = seed
        model = lgb.train(
            p,
            trn_data,
            num_boost_round=2500,
            valid_sets=[val_data],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        )

        fold_val_preds += model.predict(X_va, num_iteration=model.best_iteration) / len(SEEDS)
        fold_test_preds += model.predict(X_test, num_iteration=model.best_iteration) / len(SEEDS)

        fold_importance = pd.DataFrame({
            "feature": X_tr.columns,
            "importance": model.feature_importance(importance_type="gain"),
            "fold": fold + 1
        })
        lgb_feature_importances = pd.concat([lgb_feature_importances, fold_importance], axis=0)

    lgb_oof_probs[val_idx] = fold_val_preds
    lgb_test_probs += fold_test_preds / N_SPLITS

    fold_auc = roc_auc_score(y_va, fold_val_preds)
    fold_prauc = average_precision_score(y_va, fold_val_preds)
    lgb_fold_aucs.append(fold_auc)
    lgb_fold_praucs.append(fold_prauc)
    print(f"LGB Fold {fold + 1:02d} | ROC-AUC: {fold_auc:.6f} | PR-AUC: {fold_prauc:.6f} | Best Iter: {model.best_iteration}")

lgb_training_duration = time.time() - t0_lgb
lgb_overall_auc = roc_auc_score(y_train, lgb_oof_probs)
lgb_overall_prauc = average_precision_score(y_train, lgb_oof_probs)

print("-" * 65)
print(f"Shallow LightGBM {N_SPLITS}-Fold Mean ROC-AUC: {np.mean(lgb_fold_aucs):.6f} (+/- {np.std(lgb_fold_aucs):.6f})")
print(f"Shallow LightGBM Overall OOF ROC-AUC:  {lgb_overall_auc:.6f}")
print(f"Shallow LightGBM Overall OOF PR-AUC:   {lgb_overall_prauc:.6f}")
print(f"Training completed in:                {lgb_training_duration:.2f} seconds")
print("-" * 65)

# 5. Model 2: Shallow XGBoost (`max_depth=3`)
---
We configure XGBoost with **shallow depth-wise trees (`max_depth=3`)**. Depth 3 builds level-wise trees with at most $2^3 = 8$ terminal leaves, mirroring the shallow regularization of LightGBM while maintaining architectural diversity.

In [ ]:
import xgboost as xgb

xgb_oof_probs = np.zeros(len(X_train))
xgb_test_probs = np.zeros(len(X_test))
xgb_fold_aucs = []
xgb_fold_praucs = []
xgb_feature_importances = pd.DataFrame()

xgb_base_params = {
    "n_estimators": 2500,
    "learning_rate": 0.05,
    "max_depth": 3,            # Shallow level-wise tree growth (max 8 leaves)
    "min_child_weight": 30,
    "subsample": 0.85,
    "colsample_bytree": 0.80,
    "tree_method": "hist",
    "enable_categorical": True,
    "early_stopping_rounds": 50,
    "eval_metric": "auc",
    "n_jobs": N_JOBS,
}

print(f"Starting Shallow XGBoost {N_SPLITS}-Fold Cross-Validation (max_depth=3 | Seeds: {SEEDS})...")
t0_xgb = time.time()

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[trn_idx], y_train[trn_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]

    fold_val_preds = np.zeros(len(val_idx))
    fold_test_preds = np.zeros(len(X_test))

    for seed in SEEDS:
        p = xgb_base_params.copy()
        p["random_state"] = seed
        clf = xgb.XGBClassifier(**p)
        clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

        fold_val_preds += clf.predict_proba(X_va)[:, 1] / len(SEEDS)
        fold_test_preds += clf.predict_proba(X_test)[:, 1] / len(SEEDS)

        fold_importance = pd.DataFrame({
            "feature": X_tr.columns,
            "importance": clf.feature_importances_,
            "fold": fold + 1
        })
        xgb_feature_importances = pd.concat([xgb_feature_importances, fold_importance], axis=0)

    xgb_oof_probs[val_idx] = fold_val_preds
    xgb_test_probs += fold_test_preds / N_SPLITS

    fold_auc = roc_auc_score(y_va, fold_val_preds)
    fold_prauc = average_precision_score(y_va, fold_val_preds)
    xgb_fold_aucs.append(fold_auc)
    xgb_fold_praucs.append(fold_prauc)
    print(f"XGB Fold {fold + 1:02d} | ROC-AUC: {fold_auc:.6f} | PR-AUC: {fold_prauc:.6f} | Best Iter: {clf.best_iteration}")

xgb_training_duration = time.time() - t0_xgb
xgb_overall_auc = roc_auc_score(y_train, xgb_oof_probs)
xgb_overall_prauc = average_precision_score(y_train, xgb_oof_probs)

print("-" * 65)
print(f"Shallow XGBoost {N_SPLITS}-Fold Mean ROC-AUC: {np.mean(xgb_fold_aucs):.6f} (+/- {np.std(xgb_fold_aucs):.6f})")
print(f"Shallow XGBoost Overall OOF ROC-AUC:  {xgb_overall_auc:.6f}")
print(f"Shallow XGBoost Overall OOF PR-AUC:   {xgb_overall_prauc:.6f}")
print(f"Training completed in:                {xgb_training_duration:.2f} seconds")
print("-" * 65)

# 6. Model 3: Shallow CatBoost (`depth=4`)
---
CatBoost introduces **symmetric / oblivious decision trees** that split data along balanced hyperplanes. By setting `depth=4`, CatBoost builds balanced 16-leaf trees that provide genuine structural diversity when combined with LightGBM and XGBoost.

In [ ]:
from catboost import CatBoostClassifier

cat_oof_probs = np.zeros(len(X_train))
cat_test_probs = np.zeros(len(X_test))
cat_fold_aucs = []
cat_fold_praucs = []
cat_feature_importances = pd.DataFrame()

CAT_TASK_TYPE = "GPU" if (IS_KAGGLE and DEVICE == "gpu") else "CPU"
CAT_ITERATIONS = 2000 if IS_KAGGLE else 600

cat_base_params = {
    "iterations": CAT_ITERATIONS,
    "learning_rate": 0.05,
    "depth": 4,               # Shallow symmetric oblivious depth (16 leaves)
    "l2_leaf_reg": 5.0,
    "eval_metric": "AUC",
    "task_type": CAT_TASK_TYPE,
    "early_stopping_rounds": 50,
    "verbose": False,
}

if CAT_TASK_TYPE == "CPU":
    cat_base_params["thread_count"] = N_JOBS

X_train_cat = X_train.copy()
X_test_cat = X_test.copy()
for c in categorical_features:
    X_train_cat[c] = X_train_cat[c].astype(str)
    X_test_cat[c] = X_test_cat[c].astype(str)

print(f"Starting Shallow CatBoost {N_SPLITS}-Fold Cross-Validation (Task: {CAT_TASK_TYPE} | Depth: 4 | Seeds: {SEEDS})...")
t0_cat = time.time()

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train_cat, y_train)):
    X_tr, y_tr = X_train_cat.iloc[trn_idx], y_train[trn_idx]
    X_va, y_va = X_train_cat.iloc[val_idx], y_train[val_idx]

    fold_val_preds = np.zeros(len(val_idx))
    fold_test_preds = np.zeros(len(X_test_cat))

    for seed in SEEDS:
        p = cat_base_params.copy()
        p["random_seed"] = seed
        cb_clf = CatBoostClassifier(**p)
        cb_clf.fit(X_tr, y_tr, eval_set=(X_va, y_va), cat_features=categorical_features, verbose=False)

        fold_val_preds += cb_clf.predict_proba(X_va)[:, 1] / len(SEEDS)
        fold_test_preds += cb_clf.predict_proba(X_test_cat)[:, 1] / len(SEEDS)

        fold_importance = pd.DataFrame({
            "feature": X_tr.columns,
            "importance": cb_clf.get_feature_importance(),
            "fold": fold + 1
        })
        cat_feature_importances = pd.concat([cat_feature_importances, fold_importance], axis=0)

    cat_oof_probs[val_idx] = fold_val_preds
    cat_test_probs += fold_test_preds / N_SPLITS

    fold_auc = roc_auc_score(y_va, fold_val_preds)
    fold_prauc = average_precision_score(y_va, fold_val_preds)
    cat_fold_aucs.append(fold_auc)
    cat_fold_praucs.append(fold_prauc)
    print(f"CatBoost Fold {fold + 1:02d} | ROC-AUC: {fold_auc:.6f} | PR-AUC: {fold_prauc:.6f} | Best Iter: {cb_clf.get_best_iteration()}")

cat_training_duration = time.time() - t0_cat
cat_overall_auc = roc_auc_score(y_train, cat_oof_probs)
cat_overall_prauc = average_precision_score(y_train, cat_oof_probs)

print("-" * 65)
print(f"Shallow CatBoost {N_SPLITS}-Fold Mean ROC-AUC: {np.mean(cat_fold_aucs):.6f} (+/- {np.std(cat_fold_aucs):.6f})")
print(f"Shallow CatBoost Overall OOF ROC-AUC:  {cat_overall_auc:.6f}")
print(f"Shallow CatBoost Overall OOF PR-AUC:   {cat_overall_prauc:.6f}")
print(f"Training completed in:                 {cat_training_duration:.2f} seconds")
print("-" * 65)

# 7. Multi-Strategy Ensembling & Stacking Benchmark
---
We benchmark **5 distinct ensembling paradigms** on out-of-fold predictions to identify the highest-scoring combination:
1. **Equal 3-Way Linear Blend**: $\frac{1}{3}(P_{\text{LGB}} + P_{\text{XGB}} + P_{\text{CAT}})$.
2. **SciPy SLSQP Probability Space Optimization**: Convex weighting minimizing negative ROC-AUC.
3. **SciPy SLSQP Logit Space Optimization**: Unbounded logit space optimization with sigmoid reconstruction.
4. **3-Way Rank Averaging**: Uniform percentile ranking neutralizing calibration divergence.
5. **Stacking Meta-Learner**: $L_2$-regularized Logistic Regression meta-model on out-of-fold logits.

In [ ]:
from scipy.optimize import minimize
from scipy.special import expit, logit
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression

EPS = 1e-6
lgb_oof_c = np.clip(lgb_oof_probs, EPS, 1 - EPS)
xgb_oof_c = np.clip(xgb_oof_probs, EPS, 1 - EPS)
cat_oof_c = np.clip(cat_oof_probs, EPS, 1 - EPS)

lgb_test_c = np.clip(lgb_test_probs, EPS, 1 - EPS)
xgb_test_c = np.clip(xgb_test_probs, EPS, 1 - EPS)
cat_test_c = np.clip(cat_test_probs, EPS, 1 - EPS)

lgb_logits_oof = logit(lgb_oof_c)
xgb_logits_oof = logit(xgb_oof_c)
cat_logits_oof = logit(cat_oof_c)

lgb_logits_test = logit(lgb_test_c)
xgb_logits_test = logit(xgb_test_c)
cat_logits_test = logit(cat_test_c)

# 1. Equal 3-Way Linear Blend
oof_equal = (lgb_oof_probs + xgb_oof_probs + cat_oof_probs) / 3.0
auc_equal = roc_auc_score(y_train, oof_equal)
prauc_equal = average_precision_score(y_train, oof_equal)

# 2. SLSQP Probability Optimization
def linear_obj(weights):
    w1, w2, w3 = weights
    pred = w1 * lgb_oof_probs + w2 * xgb_oof_probs + w3 * cat_oof_probs
    return -roc_auc_score(y_train, pred)

init_w = [1/3, 1/3, 1/3]
bounds = [(0.0, 1.0)] * 3
constraints = ({"type": "eq", "fun": lambda w: sum(w) - 1.0})
opt_linear = minimize(linear_obj, init_w, method="SLSQP", bounds=bounds, constraints=constraints)
w_lin = opt_linear.x
oof_linear_opt = w_lin[0] * lgb_oof_probs + w_lin[1] * xgb_oof_probs + w_lin[2] * cat_oof_probs
auc_linear_opt = roc_auc_score(y_train, oof_linear_opt)
prauc_linear_opt = average_precision_score(y_train, oof_linear_opt)

# 3. SLSQP Logit Space Optimization
def logit_obj(weights):
    w1, w2, w3 = weights
    blend_logits = w1 * lgb_logits_oof + w2 * xgb_logits_oof + w3 * cat_logits_oof
    return -roc_auc_score(y_train, expit(blend_logits))

opt_logit = minimize(logit_obj, init_w, method="SLSQP", bounds=bounds, constraints=constraints)
w_log = opt_logit.x
oof_logit_opt = expit(w_log[0] * lgb_logits_oof + w_log[1] * xgb_logits_oof + w_log[2] * cat_logits_oof)
auc_logit_opt = roc_auc_score(y_train, oof_logit_opt)
prauc_logit_opt = average_precision_score(y_train, oof_logit_opt)

# 4. 3-Way Rank Averaging
lgb_rank = rankdata(lgb_oof_probs) / len(lgb_oof_probs)
xgb_rank = rankdata(xgb_oof_probs) / len(xgb_oof_probs)
cat_rank = rankdata(cat_oof_probs) / len(cat_oof_probs)
oof_rank = (lgb_rank + xgb_rank + cat_rank) / 3.0
auc_rank = roc_auc_score(y_train, oof_rank)
prauc_rank = average_precision_score(y_train, oof_rank)

# 5. Stacking Meta-Learner (Logistic Regression on Logits)
X_oof_meta = np.column_stack([lgb_logits_oof, xgb_logits_oof, cat_logits_oof])
meta_model = LogisticRegression(C=1.0, random_state=GLOBAL_SEED, solver="lbfgs")
meta_model.fit(X_oof_meta, y_train)
oof_stack = meta_model.predict_proba(X_oof_meta)[:, 1]
auc_stack = roc_auc_score(y_train, oof_stack)
prauc_stack = average_precision_score(y_train, oof_stack)

ensemble_results = pd.DataFrame({
    "Strategy / Architecture": [
        "Model 1: Shallow LightGBM (Single)",
        "Model 2: Shallow XGBoost (Single)",
        "Model 3: Shallow CatBoost (Single)",
        "Ensemble 1: Equal 3-Way Linear Blend",
        "Ensemble 2: SLSQP Probability Blend",
        "Ensemble 3: SLSQP Logit-Space Blend",
        "Ensemble 4: 3-Way Rank Averaging",
        "Ensemble 5: Stacking Meta-Learner (Logits)"
    ],
    "OOF ROC-AUC": [
        lgb_overall_auc,
        xgb_overall_auc,
        cat_overall_auc,
        auc_equal,
        auc_linear_opt,
        auc_logit_opt,
        auc_rank,
        auc_stack
    ],
    "OOF PR-AUC": [
        lgb_overall_prauc,
        xgb_overall_prauc,
        cat_overall_prauc,
        prauc_equal,
        prauc_linear_opt,
        prauc_logit_opt,
        prauc_rank,
        prauc_stack
    ]
}).sort_values(by="OOF ROC-AUC", ascending=False).reset_index(drop=True)

print("=" * 70)
print("SHALLOW TRI-MODEL ENSEMBLE BENCHMARK RESULTS")
print("=" * 70)
display(ensemble_results)
print("=" * 70)
print(f"SLSQP Probability Weights: [LGB: {w_lin[0]:.3f}, XGB: {w_lin[1]:.3f}, CAT: {w_lin[2]:.3f}]")
print(f"SLSQP Logit Weights:       [LGB: {w_log[0]:.3f}, XGB: {w_log[1]:.3f}, CAT: {w_log[2]:.3f}]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

pred_matrix = pd.DataFrame({
    "LightGBM": lgb_oof_probs,
    "XGBoost": xgb_oof_probs,
    "CatBoost": cat_oof_probs
})
sns.heatmap(pred_matrix.corr(), annot=True, fmt=".4f", cmap="YlGnBu", cbar=True, ax=axes[0],
            annot_kws={"fontsize": 12, "fontweight": "bold"})
axes[0].set_title("OOF Prediction Correlation Matrix", fontsize=12, fontweight="bold")

best_strategy = ensemble_results.iloc[0]["Strategy / Architecture"]
best_auc_score = ensemble_results.iloc[0]["OOF ROC-AUC"]

if "Stacking" in best_strategy:
    best_oof_preds = oof_stack
elif "Logit" in best_strategy:
    best_oof_preds = oof_logit_opt
elif "Probability" in best_strategy:
    best_oof_preds = oof_linear_opt
elif "Rank" in best_strategy:
    best_oof_preds = oof_rank
else:
    best_oof_preds = oof_equal

fpr_lgb, tpr_lgb, _ = roc_curve(y_train, lgb_oof_probs)
fpr_xgb, tpr_xgb, _ = roc_curve(y_train, xgb_oof_probs)
fpr_cat, tpr_cat, _ = roc_curve(y_train, cat_oof_probs)
fpr_ens, tpr_ens, _ = roc_curve(y_train, best_oof_preds)

axes[1].plot(fpr_lgb, tpr_lgb, label=f"LightGBM (AUC = {lgb_overall_auc:.5f})", color=PALETTE["neutral"], linewidth=1.5)
axes[1].plot(fpr_xgb, tpr_xgb, label=f"XGBoost  (AUC = {xgb_overall_auc:.5f})", color=PALETTE["accent"], linewidth=1.5)
axes[1].plot(fpr_cat, tpr_cat, label=f"CatBoost (AUC = {cat_overall_auc:.5f})", color=PALETTE["legitimate"], linewidth=1.5)
axes[1].plot(fpr_ens, tpr_ens, label=f"{best_strategy} (AUC = {best_auc_score:.5f})", color=PALETTE["highlight"], linewidth=2.5, linestyle="--")
axes[1].plot([0, 1], [0, 1], "k:", alpha=0.5, label="Chance")

axes[1].set_title("Comparative Out-Of-Fold ROC Curves", fontsize=12, fontweight="bold")
axes[1].set_xlabel("False Positive Rate (1 - Specificity)", fontsize=10)
axes[1].set_ylabel("True Positive Rate (Sensitivity)", fontsize=10)
axes[1].legend(loc="lower right", fontsize=9.5)
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.suptitle("Tri-Model Ensembling Benchmark Diagnostics", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# 8. Final Submission & Integrity Verification
---
We generate final test set predictions using the winning ensemble strategy from cross-validation and run defensive integrity assertions:
1. Exact row count match ($299,984$ rows matching `sample_submission.csv`).
2. Zero NaN / null prediction assertion.
3. Strict bounded probability range assertion ($0.0 \le P \le 1.0$).
4. Cross-partition probability calibration density check.

In [ ]:
print(f"Generating test predictions using winning strategy: '{best_strategy}'...")

if "Stacking" in best_strategy:
    X_test_meta = np.column_stack([lgb_logits_test, xgb_logits_test, cat_logits_test])
    final_test_predictions = meta_model.predict_proba(X_test_meta)[:, 1]
elif "Logit" in best_strategy:
    final_test_predictions = expit(w_log[0] * lgb_logits_test + w_log[1] * xgb_logits_test + w_log[2] * cat_logits_test)
elif "Probability" in best_strategy:
    final_test_predictions = w_lin[0] * lgb_test_probs + w_lin[1] * xgb_test_probs + w_lin[2] * cat_test_probs
elif "Rank" in best_strategy:
    test_lgb_rank = rankdata(lgb_test_probs) / len(lgb_test_probs)
    test_xgb_rank = rankdata(xgb_test_probs) / len(xgb_test_probs)
    test_cat_rank = rankdata(cat_test_probs) / len(cat_test_probs)
    final_test_predictions = (test_lgb_rank + test_xgb_rank + test_cat_rank) / 3.0
else:
    final_test_predictions = (lgb_test_probs + xgb_test_probs + cat_test_probs) / 3.0

final_submission = pd.DataFrame({
    "id": test_ids,
    "is_fraud": final_test_predictions
})

assert len(final_submission) == len(submission_df), f"Shape mismatch: {len(final_submission)} vs {len(submission_df)}"
assert not final_submission.isnull().any().any(), "Submission contains null/NaN values!"
assert (final_submission["is_fraud"] >= 0.0).all() and (final_submission["is_fraud"] <= 1.0).all(), "Predictions outside valid [0, 1] range!"
assert list(final_submission.columns) == ["id", "is_fraud"], f"Invalid columns: {list(final_submission.columns)}"

output_path = "submission.csv"
final_submission.to_csv(output_path, index=False)
print(f"Successfully generated and verified '{output_path}'!")
print(f"Submission Shape: {final_submission.shape}")
display(final_submission.head(10))

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(best_oof_preds, label="OOF Predictions (Train)", color=PALETTE["legitimate"], fill=True, alpha=0.35, linewidth=2.0)
sns.kdeplot(final_test_predictions, label="Test Predictions", color=PALETTE["fraud"], fill=True, alpha=0.35, linewidth=2.0)

plt.title("Probability Calibration Check: Train OOF vs. Test Predictions", fontsize=12, fontweight="bold")
plt.xlabel("Predicted Fraud Probability", fontsize=10)
plt.ylabel("Kernel Density", fontsize=10)
plt.legend(loc="upper right", fontsize=10)
plt.xlim(-0.02, 1.02)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

calib_summary = pd.DataFrame({
    "Partition": ["Train Out-Of-Fold", "Test Predictions"],
    "Mean Probability": [best_oof_preds.mean(), final_test_predictions.mean()],
    "Median Probability": [np.median(best_oof_preds), np.median(final_test_predictions)],
    "Std Deviation": [best_oof_preds.std(), final_test_predictions.std()],
    "Min Probability": [best_oof_preds.min(), final_test_predictions.min()],
    "Max Probability": [best_oof_preds.max(), final_test_predictions.max()]
})
display(calib_summary.round(5))